# Занятие 5. Свёртка и фильтры

**Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026**
Лекция-опора: **L5 «Пространственная фильтрация»** · **90 минут в классе** (последние 20 — клиника ДЗ 1) · ведёт ассистент

---

Сегодня пишем **свёртку своими руками** — два цикла, окно, ядро — и сверяем её с `cv2.filter2D` до 10⁻⁶.
Потом ту же сумму считаем **двумя одномерными проходами** и измеряем, что даёт сепарабельность: `2k`
умножений на пиксель вместо `k²`. В третьей части заполняем таблицу лекции **«шум × фильтр → PSNR»**
и добавляем в неё свои фильтры: какой фильтр под какой шум — решает число, а не привычка.

Заготовка начинается на слайдах 33 и 35 лекции («Окно 3 × 3» и «Корреляция и свёртка») и на демо 1.1.

| Минуты | Что происходит |
|--------|----------------|
| 0–12 | Ассистент разбирает опорный пример: окно 3 × 3 и «корреляция против свёртки» |
| 12–58 | Вы делаете **TODO 1–3** |
| 58–67 | Разбор решения |
| 67–87 | **Клиника ДЗ 1 «Фотолаборатория»**: что из занятий 1–5 идёт в ДЗ, `uint8`, мягкий край маски |
| 87–90 | Зачёт |

Ничего сдавать не нужно: **зачёт ставится в классе** по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt   (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Всё лежит в репозитории — скачивать нечего. Фото здания ИУ `lectures/L05-filtering/demo/iu-building.jpg`
(1920 × 1279) читается **в сером**, как на лекции; крупные планы — кроп лестницы 256 × 256
(`photo[880:1136, 1300:1556]`): ступени, прутья перил и ровный камень в одном кадре.

Шум трёх моделей (слайд 43) порождается здесь же, с теми же зёрнами генератора, что в демо лекции, —
поэтому числа вашей таблицы совпадут со слайдом 44 до десятой:

- **гауссов**, σ = 25 — шум чтения и усиления (высокое ISO); от яркости не зависит;
- **пуассоновский** (дробовой) — σ² ∝ I: в тенях 9 кодов, в светах 26 — та же кривая, что вы строили на занятии 2;
- **соль и перец**, 5 % — битые пиксели: 2.5 % чёрных и 2.5 % белых.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена (работает всегда,
числа другие). Номера слайдов в ноутбуке — **страницы файла** `lectures/L05-filtering/slides.pdf`
(номер в подвале самого слайда может отличаться на 1–3).

In [ ]:
LECTURE_DIR = "lectures/L05-filtering/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,                     # публичный репозиторий: labs/lab05-filters/
                  "../../../iu-intro-cv/" + LECTURE_DIR,      # ноутбук открыт из iu-intro-cv-ta
                  "data"]                                     # уже скачанное
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(880, 1136), slice(1300, 1556))                  # лестница с перилами, 256 × 256


def synthetic_photo(h=1279, w=1920):
    """Запас без сети: плавный фон, три круга, «ступени» и «прутья» на месте кропа. Числа будут другие."""
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[880:1136:16, 1300:1556] -= 70
    img[880:1136, 1300:1556:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    """Фото лекции: (серое, цветное BGR, откуда взято)."""
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), cio.imread(p), d
    try:                                                   # Colab: репозитория рядом нет — качаем
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), cio.imread(p), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        s = synthetic_photo()
        return s, cv2.cvtColor(s, cv2.COLOR_GRAY2BGR), "синтетика"


def add_noise(img):
    """Три модели шума с зёрнами лекции (демо 1.2 и 2.1) — числа таблицы совпадают со слайдом 44."""
    gauss = np.clip(img + np.random.default_rng(0).normal(0, 25, img.shape), 0, 255).astype(np.uint8)
    poisson = np.clip(np.random.default_rng(1).poisson(img / 4.0) * 4, 0, 255).astype(np.uint8)
    u = np.random.default_rng(2).random(img.shape); sp = img.copy()
    sp[u < 0.025] = 0; sp[u > 0.975] = 255                                 # соль и перец, 5 %
    return {"Гаусс σ 25": gauss, "Пуассон": poisson, "соль-перец 5 %": sp}


photo, photo_bgr, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8, "ожидался серый кадр 1920 × 1279, uint8"
crop = photo[CROP]
NOISES = add_noise(photo)
print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· кроп", crop.shape)
for name, im in NOISES.items():
    print("   %-15s PSNR к чистому кадру %.1f дБ" % (name, metrics.psnr(photo, im)))
viz.grid({"кроп лестницы 256 × 256": crop, **{name: im[CROP] for name, im in NOISES.items()}}, cols=4, size=3.2)

## 2. Опорный пример — разбирает ассистент

Здесь всё написано, задача — **понять каждую строку**.

**Фильтр — это скалярное произведение окна и ядра.** Слайд 33: окно 3 × 3 вокруг пикселя на краю ступени,
значения `[[32, 30, 32], [140, 130, 118], [253, 255, 255]]`. Умножаем поэлементно на ядро и складываем:
box даёт 138.3, Гаусс `[1 2 1]ᵀ · [1 2 1] / 16` — 136.1. Оба ядра в сумме дают единицу, поэтому на ровном
месте яркость не меняется; отличие — в весе центра, 1/9 против 4/16.

**`cv2.filter2D(img, ddepth, kernel)`** делает то же для каждого пикселя кадра. `ddepth = -1` — тип как
у входа (`uint8`, с округлением и насыщением); `cv2.CV_64F` — вещественный результат как есть.

In [ ]:
PY, PX = 1035, 1426                                           # пиксель на краю ступени (слайд 33): строка, столбец
win = photo[PY - 1:PY + 2, PX - 1:PX + 2].astype(np.float64)  # окно 3 × 3 вокруг него
box3 = np.ones((3, 3)) / 9.0
gauss3 = np.outer([1, 2, 1], [1, 2, 1]) / 16.0
print("окно 3 × 3:", win.astype(int).tolist())
print("сумма весов: box %.3f, Гаусс %.3f · вес центра: %.3f против %.3f" % (box3.sum(), gauss3.sum(), box3[1, 1], gauss3[1, 1]))
print("руками:    box %.1f · Гаусс %.1f" % ((win * box3).sum(), (win * gauss3).sum()))
f64 = photo.astype(np.float64)
print("filter2D:  box %.1f · Гаусс %.1f   ← то же число в том же пикселе"
      % (cv2.filter2D(f64, cv2.CV_64F, box3)[PY, PX], cv2.filter2D(f64, cv2.CV_64F, gauss3)[PY, PX]))

### Корреляция и свёртка — не одно и то же

Демо 1.1 и слайд 35. Возьмём ядро «единица справа на 2». **Корреляция** `g(x, y) = Σ h(i, j) · I(x + i, y + j)`
берёт пиксель *справа* — кадр уезжает влево. **Свёртка** `g(x, y) = Σ h(i, j) · I(x − i, y − j)` — то же
с ядром, перевёрнутым по обеим осям: кадр уезжает вправо. `cv2.filter2D` считает **корреляцию**; нужна
свёртка — переверните ядро, `cv2.flip(k, -1)`. Для симметричных ядер (box, Гаусс) разницы нет — поэтому
её так легко не заметить.

In [ ]:
k = np.zeros((5, 5), np.float32); k[2, 4] = 1             # «взять пиксель справа на 2»
corr = cv2.filter2D(photo, -1, k)                         # filter2D — корреляция
conv = cv2.filter2D(photo, -1, cv2.flip(k, -1))           # свёртка = перевёрнутое ядро
ref = np.zeros_like(photo); ref[:, 2:] = photo[:, :-2]    # свёртка руками: сдвиг вправо на 2
d = lambda a, b: int(np.abs(a.astype(int) - b)[10:-10, 10:-10].max())
print("filter2D против свёртки: %d кодов; с flip: %d" % (d(corr, ref), d(conv, ref)))
v = cv2.getGaussianKernel(5, 1.0); gk = v @ v.T           # симметричное ядро 5 × 5
print("Гаусс: %d — перевёрнутое симметричное ядро равно самому себе" % d(cv2.filter2D(photo, -1, gk), cv2.filter2D(photo, -1, cv2.flip(gk, -1))))

### Вопрос, на который отвечаем вслух

Свёрточная сеть называется «свёрточной», но слой `Conv2d` в PyTorch ядро **не переворачивает** — считает
корреляцию, как `filter2D`. Почему для ядра, которое *обучается*, это неважно, а для ядра, которое вы
написали руками (сдвиг, производная), — важно?

---

## TODO 1 — свёртка своими руками *(≈ 12 минут)*

Напишите `conv2d(img, kernel)` — **свёртку** (не корреляцию) кадра с ядром нечётного размера `kh × kw`.
Результат — `float64` того же размера, что кадр; граница — `BORDER_REFLECT_101`, умолчание OpenCV.

Скелет — три шага:

1. перевернуть ядро по обеим осям: `kernel[::-1, ::-1]` — в этом вся разница между свёрткой и корреляцией;
2. достроить кадр на полядра с каждой стороны:
   `cv2.copyMakeBorder(img, kh // 2, kh // 2, kw // 2, kw // 2, cv2.BORDER_REFLECT_101)` — порядок: верх, низ, лево, право;
3. два вложенных цикла по `y` и `x`: окно `pad[y:y + kh, x:x + kw]` умножить поэлементно на перевёрнутое
   ядро и сложить — одно скалярное произведение на пиксель.

Считаем на кропе 256 × 256: это 65 536 итераций цикла Python, доли секунды. Почему так медленно и что
с этим делать — в ячейке после ассертов и в ⭐.

> **Подсказка.** `(a * b).sum()` — поэлементное произведение и сумма. В достроенном кадре окно вокруг
> пикселя `(y, x)` начинается в точке `(y, x)`: сдвиг на полядра уже сделала достройка.

In [ ]:
def conv2d(img, kernel):
    """Свёртка (не корреляция) кадра с ядром kh × kw нечётного размера → float64 того же размера; граница BORDER_REFLECT_101."""
    img, kernel = np.asarray(img, np.float64), np.asarray(kernel, np.float64)
    kh, kw = kernel.shape
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: перевернуть ядро [::-1, ::-1]; copyMakeBorder на kh // 2 и kw // 2 (BORDER_REFLECT_101); два цикла по y, x: out[y, x] = (окно * ядро).sum()
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


K_TEST = np.float64([[1, 2, 0], [0, 3, -1], [-2, 0, 1]])                      # нарочно несимметричное
flt = lambda im, kern, border=cv2.BORDER_REFLECT_101: cv2.filter2D(np.float64(im), cv2.CV_64F, np.float64(kern), borderType=border)
mine = conv2d(crop, K_TEST)
assert isinstance(mine, np.ndarray) and mine.shape == crop.shape, "результат — массив того же размера, что кадр"
assert np.abs(mine - flt(crop, K_TEST)).max() > 1, "получилась корреляция: ядро не перевёрнуто — kernel[::-1, ::-1]"
ref = flt(crop, cv2.flip(K_TEST, -1))                                         # свёртка через filter2D: перевёрнутое ядро
assert np.abs(mine - ref)[8:-8, 8:-8].max() < 1e-6, "внутри кадра расхождение с filter2D %.3g: проверьте срез окна pad[y:y + kh, x:x + kw]" % np.abs(mine - ref)[8:-8, 8:-8].max()
assert np.abs(mine - ref).max() < 1e-6, "внутри кадра верно, у края — нет (%.3g): достройка должна быть BORDER_REFLECT_101" % np.abs(mine - ref).max()
K_RECT = np.arange(21, dtype=np.float64).reshape(3, 7) - 8                    # ядро 3 × 7: kh ≠ kw
assert np.abs(conv2d(crop, K_RECT) - flt(crop, cv2.flip(K_RECT, -1))).max() < 1e-6, "ядро 3 × 7: высота окна — kh, ширина — kw; не перепутаны?"
shift = np.zeros((5, 5)); shift[2, 4] = 1                                     # ядро опорного примера
assert np.array_equal(conv2d(crop, shift)[:, 2:], crop[:, :-2]), "свёртка с ядром «единица справа на 2» сдвигает кадр вправо на 2"
print("TODO 1 ✔  · расхождение с filter2D (перевёрнутое ядро): %.1e · с корреляцией: %.0f кодов" % (np.abs(mine - ref).max(), np.abs(mine - flt(crop, K_TEST)).max()))

### Сколько стоит цикл на Python

Ваша `conv2d` и `cv2.filter2D` считают одно и то же число. Разница — во времени: измерим на кропе
и оценим, сколько заняла бы ваша функция на полном кадре 1920 × 1279.

In [ ]:
ms = lambda fn, repeat=3: 1e3 * min(timeit.repeat(fn, number=1, repeat=repeat))       # лучшее из нескольких прогонов, мс
t_mine = ms(lambda: conv2d(crop, K_TEST))
t_cv = ms(lambda: flt(crop, cv2.flip(K_TEST, -1)), 5)
scale = photo.size / crop.size                                                        # во сколько раз кадр больше кропа
print("кроп 256 × 256, ядро 3 × 3: ваша conv2d %.0f мс · filter2D %.2f мс — в %.0f раз быстрее" % (t_mine, t_cv, t_mine / t_cv))
print("кадр 1920 × 1279 — пикселей больше в %.0f раз: ваша conv2d ≈ %.1f с · filter2D %.0f мс"
      % (scale, 1e-3 * t_mine * scale, ms(lambda: flt(photo, cv2.flip(K_TEST, -1)), 5)))

### Линейность: свойство, у которого есть цена

Слайд 36. Линейный фильтр не различает «сигнал» и «шум»: фильтр шумного кадра = фильтр чистого + фильтр
шума. Два Гаусса подряд — один Гаусс с σ = √(σ₁² + σ₂²). У медианы этих свойств нет — и поэтому она
умеет то, чего не умеет ни одно ядро (TODO 3).

In [ ]:
A = photo.astype(np.float32)                                         # чистый кадр
N = NOISES["Гаусс σ 25"].astype(np.float32) - A                      # шум отдельно
G2 = lambda x: cv2.GaussianBlur(x, (0, 0), 2)
M5 = lambda x: cv2.medianBlur(x, 5)
print("линейность:      |G(чистый + шум) − G(чистый) − G(шум)| ≤ %.4f кода" % np.abs(G2(A + N) - G2(A) - G2(N)).max())
two, one = cv2.GaussianBlur(cv2.GaussianBlur(A, (0, 0), 1.5), (0, 0), 2.0), cv2.GaussianBlur(A, (0, 0), 2.5)
print("ассоциативность: Гаусс σ 1.5, потом σ 2.0 против одного σ 2.5: ≤ %.3f кода" % np.abs(two - one).max())
print("медиана 5:       |M(чистый + шум) − M(чистый) − M(шум)| до %.0f кодов — фильтр нелинейный" % np.abs(M5(A + N) - M5(A) - M5(N)).max())

### Вопрос, на который отвечаем вслух

Ваша `conv2d` считает то же, что `filter2D`, и проигрывает ему в сотни раз, хотя умножений в обоих случаях
поровну — `k²` на пиксель. На что уходит время в вашей версии? (Один из ответов — звёздочка в конце ноутбука.)

---

## TODO 2 — Гаусс двумя одномерными проходами *(≈ 15 минут)*

Слайд 40. Ядро Гаусса **сепарабельно**: `K = v · vᵀ`, где `v` — одномерный столбец. Поэтому вместо одного
прохода ядром `k × k` (`k²` умножений на пиксель) делают два: ядром `1 × k` по строкам и ядром `k × 1`
по столбцам — `2k` умножений. Для `k = 31` это 62 вместо 961.

Напишите `gauss_sep(img, sigma, ksize=None)` → `float64` того же размера:

1. размер ядра — `ksize`, а если он не задан, правило **±3σ**: `2 * ceil(3σ) + 1`;
2. одномерное ядро: `v[i] = exp(−x² / 2σ²)`, где `x = i − ksize // 2`; **нормировать**: `v /= v.sum()`;
3. достроить кадр на `r = ksize // 2` со всех сторон (`BORDER_REFLECT_101`);
4. **проход по строкам** — сумма `ksize` сдвинутых копий кадра с весами: `rows += v[i] * pad[:, i:i + w]`;
5. **проход по столбцам** — то же по результату первого прохода: `out += v[i] * rows[i:i + h]`.

В TODO 1 цикл шёл **по пикселям**, и на каждом шаге NumPy умножал маленькое окно на ядро. Здесь цикл идёт
**по отводам ядра**, и на каждом шаге NumPy умножает на вес весь кадр сразу. Сумма та же, порядок
слагаемых другой — итераций цикла `2k` вместо `H · W`.

> **Подсказка.** `pad[:, i:i + w]` — кадр, сдвинутый на `i − r` столбцов. После первого прохода в `rows`
> остаётся `h + 2r` строк и `w` столбцов: достройка по вертикали нужна второму проходу.

In [ ]:
def gauss_sep(img, sigma, ksize=None):
    """Гаусс двумя одномерными проходами (строки, потом столбцы) → float64 того же размера; граница BORDER_REFLECT_101."""
    h, w = img.shape[:2]
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 9–11 строк: ksize по правилу ±3σ; v = exp(−x² / 2σ²), v /= v.sum(); copyMakeBorder на r = ksize // 2; цикл по i: rows += v[i] * pad[:, i:i + w]; второй цикл: out += v[i] * rows[i:i + h]
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


flat = np.full((40, 60), 100, np.uint8)
g_flat = np.asarray(gauss_sep(flat, 2.0), np.float64)
assert g_flat.shape == flat.shape, "результат того же размера, что кадр (сейчас %s)" % (g_flat.shape,)
assert np.abs(g_flat - 100).max() < 1e-9, "ровный кадр 100 должен остаться 100 (сейчас %.1f): ядро нормировано? v /= v.sum()" % g_flat.mean()
mine = np.asarray(gauss_sep(photo, 1.5), np.float64)                              # ksize по правилу ±3σ: 11
ref = cv2.GaussianBlur(np.float64(photo), (11, 11), 1.5)
assert np.abs(mine - ref)[20:-20, 20:-20].max() < 1e-6, "внутри кадра расхождение с GaussianBlur(σ 1.5, ядро 11 × 11) %.3g: формула ядра и размер 2 · ceil(3σ) + 1" % np.abs(mine - ref)[20:-20, 20:-20].max()
assert np.abs(mine - ref).max() < 1e-6, "внутри кадра верно, у края — нет (%.3g): достройка BORDER_REFLECT_101 на ksize // 2 со всех сторон" % np.abs(mine - ref).max()
v31 = cv2.getGaussianKernel(31, 4.0); K31 = v31 @ v31.T                           # полное ядро 31 × 31 = v · vᵀ (σ 4: по правилу ±3σ было бы 25)
d_sep = np.abs(gauss_sep(crop, 4.0, 31) - flt(crop, K31)).max()
assert d_sep < 1e-6, "два прохода 1-D должны совпасть с одним проходом ядром 31 × 31 (сейчас %.3g): ksize передан?" % d_sep
print("TODO 2 ✔  · против GaussianBlur: %.1e · два прохода против ядра 31 × 31: %.1e · против GaussianBlur на uint8: до %.1f кода"
      % (np.abs(mine - ref).max(), d_sep, np.abs(mine - cv2.GaussianBlur(photo, (11, 11), 1.5)).max()))
viz.grid({"кроп": crop, "gauss_sep σ 1.5": mine[CROP], "gauss_sep σ 4": gauss_sep(photo, 4.0)[CROP]}, cols=3, size=3.4)

### Что даёт сепарабельность — измеряем

Сначала на ваших функциях. `taps2d` ниже — полное ядро `k × k` «по отводам»: тот же приём, что в вашем
`gauss_sep`, но цикл двойной, `k²` слагаемых. Против него — ваши два прохода, `2k` слагаемых.
Потом четыре функции OpenCV на полном кадре: это слайд 41 «Порядки величин».

In [ ]:
def taps2d(img, K):
    """Полное ядро kh × kw «по отводам»: kh · kw сдвинутых копий кадра с весами (для симметричного ядра свёртка = корреляция)."""
    (kh, kw), (h, w) = K.shape, img.shape[:2]
    pad = cv2.copyMakeBorder(np.float64(img), kh // 2, kh // 2, kw // 2, kw // 2, cv2.BORDER_REFLECT_101)
    out = np.zeros((h, w))
    for i in range(kh):
        for j in range(kw):
            out += K[i, j] * pad[i:i + h, j:j + w]
    return out


print("кроп 256 × 256, NumPy:")
for k in (5, 15, 31, 61):
    v = cv2.getGaussianKernel(k, k / 6); K = v @ v.T
    t2, t1 = ms(lambda: taps2d(crop, K)), ms(lambda: gauss_sep(crop, k / 6, k))
    print("   k = %2d: %4d слагаемых %6.1f мс · %3d слагаемых %5.2f мс · выигрыш ×%.1f (k / 2 = %.1f)" % (k, k * k, t2, 2 * k, t1, t2 / t1, k / 2))
print("кадр 1920 × 1279, OpenCV, uint8:")
for k in (5, 15, 31, 61):
    v = cv2.getGaussianKernel(k, k / 6); K = (v @ v.T).astype(np.float32)
    print("   k = %2d: filter2D %5.1f мс · sepFilter2D %4.1f · GaussianBlur %4.1f · blur (box) %4.1f"
          % (k, ms(lambda: cv2.filter2D(photo, -1, K), 5), ms(lambda: cv2.sepFilter2D(photo, -1, v, v), 5),
             ms(lambda: cv2.GaussianBlur(photo, (k, k), k / 6), 5), ms(lambda: cv2.blur(photo, (k, k)), 5)))

### Вопрос, на который отвечаем вслух

Три наблюдения из таблицы OpenCV — объясните каждое. (1) `blur` стоит одинаково при `k = 5` и `k = 61`
(слайд 42). (2) `sepFilter2D` растёт вместе с `k`, но медленно. (3) `filter2D` между `k = 5` и `k = 15`
делает ступеньку, а дальше почти не растёт: большое ядро он считает уже **не суммой по окну**. Чем? (Ответ — L6.)

---

## TODO 3 — таблица «шум × фильтр → PSNR» *(≈ 12 минут)*

Слайд 44 — и обещание с него: «строки и столбцы вы заполните сами и добавите свои фильтры». Четыре фильтра:

| Фильтр | Вызов | Что делает |
|--------|-------|------------|
| box 5 × 5 | `cv2.blur(x, (5, 5))` | среднее по окну: все 25 весов равны 1/25 |
| Гаусс σ 1.5 | `cv2.GaussianBlur(x, (0, 0), 1.5)` | взвешенное среднее; `ksize = (0, 0)` — размер ядра по σ |
| медиана 5 | `cv2.medianBlur(x, 5)` | **медиана** окна 5 × 5: сортировка вместо суммы, нелинейный; выброс в окне на ответ не влияет |
| билатеральный | `cv2.bilateralFilter(x, 9, 75, 75)` | Гаусс, у которого вес соседа зависит ещё и от **разницы яркости** с центром (`sigmaColor = 75`): через резкий край не усредняет |

Напишите `noise_filter_table(clean, noises, filters)` → `(db, ms)`:

- `db` — массив `(len(noises), 1 + len(filters))`: столбец 0 — PSNR шумного кадра к чистому, дальше — PSNR
  результата каждого фильтра, дБ;
- `ms` — массив `(len(filters),)`: время каждого фильтра на первом шумном кадре, мс, минимум из трёх прогонов.

`noises` — словарь «имя → шумный кадр», `filters` — словарь «имя → функция от кадра»; порядок строк
и столбцов — порядок словарей.

> **Подсказка.** `metrics.psnr(clean, img)` — библиотека курса; `enumerate(filters.values())` — номер
> и функция; `1e3 * min(timeit.repeat(lambda: f(img), number=1, repeat=3))` — миллисекунды.

In [ ]:
FILTERS = {"box 5": lambda x: cv2.blur(x, (5, 5)),
           "Гаусс 1.5": lambda x: cv2.GaussianBlur(x, (0, 0), 1.5),
           "медиана 5": lambda x: cv2.medianBlur(x, 5),
           "билатеральный": lambda x: cv2.bilateralFilter(x, 9, 75, 75)}


def noise_filter_table(clean, noises, filters=FILTERS):
    """«Шум × фильтр» → (db, ms): db — PSNR, дБ, (шумы, 1 + фильтры), столбец 0 — без фильтра; ms — время фильтров, мс."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 7–9 строк: db = zeros((шумы, 1 + фильтры)); цикл по шумам: столбец 0 — metrics.psnr(clean, noisy), дальше — metrics.psnr(clean, f(noisy)); ms — min(timeit.repeat(...)) на первом шумном кадре
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


def print_table(db, t_ms, noises, filters):
    print("%-16s %6s" % ("PSNR, дБ", "без") + "".join("%22s" % n for n in filters))
    for name, row in zip(noises, db):
        best = 1 + int(np.argmax(row[1:]))
        print("%-16s %6.1f" % (name, row[0]) + "".join("%22s" % ("%.1f%s" % (v, " ◄" if j + 1 == best else "")) for j, v in enumerate(row[1:])))
    print("%-16s %6s" % ("время, мс", "") + "".join("%22.1f" % v for v in t_ms))


SLIDE_44 = np.array([[20.7, 24.1, 25.0, 24.8, 26.5], [21.6, 24.1, 25.0, 24.9, 26.5], [17.7, 23.7, 24.5, 26.1, 20.6]])
db, t_ms = [np.asarray(a, np.float64) for a in noise_filter_table(photo, NOISES, FILTERS)]
assert db.shape == (3, 5) and t_ms.shape == (4,), "ожидались массивы (3 шума × [без фильтра + 4 фильтра]) и (4 фильтра,), сейчас %s и %s" % (db.shape, t_ms.shape)
assert np.all(np.isfinite(db)) and np.all(t_ms > 0), "PSNR — конечные числа, время — положительное"
sp = NOISES["соль-перец 5 %"]
assert abs(db[2, 0] - metrics.psnr(photo, sp)) < 1e-6, "столбец 0 — PSNR шумного кадра без фильтра"
assert abs(db[2, 3] - metrics.psnr(photo, cv2.medianBlur(sp, 5))) < 1e-6, "строка «соль-перец», столбец «медиана 5» должна совпасть с прямым вызовом: фильтр применяется к шумному кадру, PSNR — к чистому"
assert np.all(db[:, 1:].max(1) > db[:, 0]), "лучший фильтр в каждой строке должен улучшать PSNR шумного кадра"
assert 0.05 < t_ms.min() and t_ms.max() < 5000, "время — в миллисекундах (сейчас %s)" % np.round(t_ms, 3).tolist()
if SRC != "синтетика":
    assert np.abs(db - SLIDE_44).max() < 0.15, "на фото лекции таблица должна совпасть со слайдом 44 (расхождение %.2f дБ)" % np.abs(db - SLIDE_44).max()
print("TODO 3 ✔  · ◄ — лучший фильтр в строке")
print_table(db, t_ms, NOISES, FILTERS)
noisy = NOISES["Гаусс σ 25"]
viz.grid({"шум σ 25": noisy[CROP], **{name: f(noisy)[CROP] for name, f in FILTERS.items()}}, cols=5, size=2.8)

### Добавьте свои фильтры

Таблица — инструмент: новая идея проверяется одной строкой словаря. В ячейке ниже четыре кандидата;
допишите свои — другая σ, другое окно, два фильтра подряд, ваш `gauss_sep` — и перезапустите.
NLM (`fastNlMeansDenoising`, столбец слайда 44) считается около секунды на кадр.

Своё фото (не обязательно): положите его в `data/my-photo.jpg` — ячейка построит ту же таблицу на нём.

In [ ]:
MORE = {"медиана 3": lambda x: cv2.medianBlur(x, 3),
        "Гаусс 0.75": lambda x: cv2.GaussianBlur(x, (0, 0), 0.75),
        "медиана 3 → Гаусс 1": lambda x: cv2.GaussianBlur(cv2.medianBlur(x, 3), (0, 0), 1.0),
        "NLM h = 25": lambda x: cv2.fastNlMeansDenoising(x, None, 25, 7, 21),
        # "мой фильтр": lambda x: ...,                   ← допишите свою строку
        }
db2, t2 = noise_filter_table(photo, NOISES, MORE)
print_table(db2, t2, NOISES, MORE)

MY_PHOTO = "data/my-photo.jpg"
if os.path.exists(MY_PHOTO):
    my = cio.imread(MY_PHOTO, "gray")
    print("\nсвоё фото", my.shape)
    print_table(*noise_filter_table(my, add_noise(my), FILTERS), add_noise(my), FILTERS)

### Убрать шум, не убив края

Слайд 5 — мотивация лекции. Один PSNR на кадр скрывает главное: где именно фильтр ошибается. Разделим
кадр на **края** (полоса 5 px вокруг границ чистого кадра, найденных детектором Canny) и **ровные области**
и посчитаем PSNR отдельно.

In [ ]:
edges = cv2.dilate(cv2.Canny(cv2.GaussianBlur(photo, (0, 0), 1.2), 60, 150), np.ones((5, 5), np.uint8)) > 0   # полоса 5 px вокруг краёв
psnr_m = lambda a, b, m: 10 * np.log10(255 ** 2 / np.mean((a[m].astype(float) - b[m]) ** 2))                   # PSNR по маске
print("края занимают %.0f %% кадра" % (100 * edges.mean()))
print("%-16s %8s %8s %8s" % ("шум σ 25, дБ", "кадр", "края", "ровное"))
for name, f in {"без фильтра": lambda x: x, **FILTERS, "Гаусс 0.75": MORE["Гаусс 0.75"]}.items():
    o = f(noisy)
    print("%-16s %8.1f %8.1f %8.1f" % (name, metrics.psnr(photo, o), psnr_m(photo, o, edges), psnr_m(photo, o, ~edges)))

### Вопрос, на который отвечаем вслух

Билатеральный фильтр — лучший на гауссовом шуме (26.5 дБ) и худший на соли и перце: 20.6 дБ, хуже box.
Почему фильтр, который «бережёт края», бережёт и битые пиксели? И второй вопрос: box 5 × 5 на краях даёт
19.2 дБ при 20.6 у кадра без фильтра — что именно он испортил?

---

## Клиника ДЗ 1 «Фотолаборатория» *(последние 20 минут)*

**Дедлайн — вс 11.10, 23:59.** ТЗ — `homeworks/hw1-photolab/README.md`, рубрика — `rubric.md`, ожидаемые
числа на кадрах курса — `sample_data/README.md`. Почти все одиннадцать функций шаблона вы уже писали:

| Функция шаблона | Где вы это делали | Что добавляет сегодняшнее занятие |
|-----------------|-------------------|-----------------------------------|
| `gamma_correct`, `white_balance` | L3, слайды 14–17 | считать во `float`, обратно — через `np.clip`: ячейка ниже |
| `clahe_lab` | L3, слайд 24; занятие 3 | — |
| `color_mask`, `largest_component` | занятие 3, TODO 1–2 | — |
| `chroma_key` | новое: альфа и композит | мягкий край маски — **Гаусс по альфе**: ячейка ниже |
| `rotate_keep_all` | занятие 4, «мост к ДЗ» | — |
| `resize_aa` | занятие 4, «AREA в варпе не работает» | размытие перед уменьшением — тот же `GaussianBlur` |
| `order_corners`, `rectify` | занятие 4, TODO 1–2 | — |
| `contact_sheet` | занятие 1, ⭐ | — |

**Как проверять себя.** `pytest tests -q` — 11 тестов на синтетике; потом **те же команды на кадрах курса**:
числа должны совпасть с `sample_data/README.md` до последнего знака (хромакей на `ball-lamp.jpg` — recall
94.6 %, precision 99.9 %; выпрямление `poster-angle.jpg` — 99.1 %; поворот на 30° — 2302 × 2068).
Зелёные тесты — половина работы; вторая половина — свои фото, числа и отчёт.

### `uint8` переполняется молча

Слайд 46 — и первая причина «перевёрнутых» пикселей в ДЗ. Баланс белого умножает канал на коэффициент
больше единицы, гамма и композит считают во `float` — а при переводе обратно в `uint8` число 280
**без предупреждения** превращается в 24. Правило курса: считать во `float`, в `uint8` — только через
`np.clip` (в шаблоне ДЗ это функция `as_uint8`).

In [ ]:
wrapped = photo + np.uint8(100)                                        # арифметика uint8 — по модулю 256
print("photo + 100 в uint8: %.1f %% пикселей стали темнее, чем были; 200 + 100 = %d"
      % (100 * (wrapped < photo).mean(), (np.array([200], np.uint8) + np.uint8(100))[0]))
gains = np.array([1.0, 1.1, 1.4])                                      # «баланс белого»: G усилен на 10 %, R — на 40 %
bad = (photo_bgr * gains).astype(np.uint8)                             # 280 → 24, предупреждения нет
good = np.clip(photo_bgr * gains, 0, 255).astype(np.uint8)             # float → clip → uint8
print("усиление (1.0, 1.1, 1.4) без np.clip: испорчено %.1f %% пикселей" % (100 * (bad != good).any(2).mean()))
viz.grid({"без np.clip: светлое «перевернулось»": bad, "np.clip(…, 0, 255)": good}, cols=2, size=4.5)

### Мягкий край маски — сегодняшний Гаусс

`chroma_key` заканчивается композитом `кадр · α + фон · (1 − α)`. С бинарной маской край объекта выглядит
вырезанным ножницами; размытие маски Гауссом с σ = `feather` превращает ступеньку в плавный переход шириной
≈ 2.56 σ. Три места, где здесь ошибаются: α осталась в диапазоне 0…255, а не 0…1; у α форма `(H, W)`,
а у кадра `(H, W, 3)`; фон другого размера, чем кадр.

In [ ]:
yy, xx = np.mgrid[:240, :320]
disk = np.hypot(xx - 160, yy - 120) <= 70                                  # «объект» — диск; его маска известна точно
fg = np.full((240, 320, 3), 90, np.uint8); fg[disk] = (40, 40, 220)        # кадр: красный диск на сером
bg = cv2.resize(photo_bgr, (320, 240), interpolation=cv2.INTER_AREA)       # фон приводится к размеру кадра до композита
shots = {}
for feather in (0, 2, 8):
    alpha = disk.astype(np.float32)                                        # α в диапазоне 0…1
    if feather:
        alpha = cv2.GaussianBlur(alpha, (0, 0), feather)
    comp = fg * alpha[..., None] + bg * (1 - alpha[..., None])             # [..., None]: (H, W) → (H, W, 1)
    row, xs = alpha[120, 160:][::-1], np.arange(160)[::-1]                 # профиль α от центра диска вправо
    print("feather σ = %d: переход α от 0.9 до 0.1 — %.1f px (2.56 σ = %.1f)"
          % (feather, np.interp(0.1, row, xs) - np.interp(0.9, row, xs), 2.563 * feather))
    shots["feather σ = %d" % feather] = np.clip(comp, 0, 255).astype(np.uint8)[60:180, 170:290]
viz.grid(shots, cols=3, size=3.2, suptitle="край диска: бинарная маска, σ 2 и σ 8")

### Вопросы клиники

Принесли вопрос по своему ДЗ — задавайте сейчас. Если вопросов нет, разбираем по умолчанию **хромакей
с `--sample`** (границы H, S с прямоугольника на объекте, V открыт; красный — через 0/179) и **выпрямление
по кликам** (порядок углов, размер выхода из `--aspect`). С сегодняшнего занятия ассистент принимает
**показ ДЗ**: одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка *(по желанию, +0.5, до конца следующей недели)*

**Свёртка без циклов Python.** `np.lib.stride_tricks.sliding_window_view(pad, (kh, kw))` возвращает
четырёхмерный **вид** `(H, W, kh, kw)` — все окна кадра сразу, без копирования памяти. Остаётся умножить
каждое окно на перевёрнутое ядро и сложить по двум последним осям: `np.einsum("ijkl,kl->ij", win, flipped)`.

Напишите `conv2d_fast(img, kernel)` — ячейка ниже сверит её с вашей `conv2d` и измерит время. Сдаётся
ноутбук с функцией, таблицей для `k = 3, 5, 15` и одной фразой: почему `filter2D` всё равно быстрее.

In [ ]:
def conv2d_fast(img, kernel):
    """⭐ Свёртка без циклов Python: sliding_window_view + einsum → float64 того же размера."""
    raise NotImplementedError("⭐ звёздочка: sliding_window_view + einsum")


try:
    fast = conv2d_fast(crop, K_TEST)
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть; замеры появятся, когда функция будет написана")
else:
    assert np.abs(fast - conv2d(crop, K_TEST)).max() < 1e-9, "conv2d_fast должна совпасть с conv2d до 1e-9"
    for k in (3, 5, 15):
        kern = np.random.default_rng(5).normal(0, 1, (k, k))
        t_loop, t_fast, t_cv = ms(lambda: conv2d(crop, kern)), ms(lambda: conv2d_fast(crop, kern)), ms(lambda: flt(crop, cv2.flip(kern, -1)), 5)
        print("k = %2d: циклы %4.0f мс · conv2d_fast %5.2f мс (×%.0f) · filter2D %.2f мс (ещё ×%.0f)" % (k, t_loop, t_fast, t_loop / t_fast, t_cv, t_fast / t_cv))

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–3 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **чем свёртка отличается от корреляции** и что из двух считает `cv2.filter2D`;
- [ ] на вопросы «почему два прохода `1 × k` и `k × 1` дают тот же результат, что ядро `k × k`» и «почему
      под соль и перец — медиана, а не Гаусс» есть внятный ответ.

**2 % итоговой оценки**, ставится в классе. Досылать ничего не нужно.